# 1.3 DFL 데드볼 선수 모델 비교

**입력** · `data/external/dfl/`의 공개 DFL 7경기와 v2 특징 캐시. 대회 경기는 사용하지 않는다.

**분할** · 6경기 학습, 나머지 1경기 평가를 7번 반복. 학습은 HSJ 경기별 랜덤 노이즈, 검증은 HSJ 고정 노이즈. 비터비와 라인아웃 규칙은 HSJ 평가와 동일하다.

**실행** · Python 3.12 커널에 dead_ball/requirements.txt 및 `xgboost==3.4.1`, `catboost==1.2.10`, `torch` 설치. macOS에서 LightGBM·XGBoost에 OpenMP 런타임 필요. 셀을 순서대로 실행한다.

**출력** · `model/영상추출/추출csv/dead_ball/dfl_lomo_20261111_01/`에 경기별 점수, 모델별 요약, 매니페스트, 검증 결과. 한 실행ID에는 동일한 데이터·설정만 사용한다.

In [ ]:
import hashlib
from importlib.metadata import version
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
from time import perf_counter

import numpy as np
import pandas as pd
from scipy.special import expit

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/train_dfl.py").is_file()
)
sys.path.insert(0, str(ROOT / "src"))
import compare_smoothing as scoring
from config import CONFIG
import dfl_labels
import dfl_load
import features
import final_dead_balls
import metrics
import smoothing
import train_dfl

MATCHES = tuple(dfl_load.MATCHES)
FPS = CONFIG["fps"]
STRIDE = CONFIG["train_stride"]
SEED = CONFIG["sim_seed"]
THREADS = min(6, os.cpu_count() or 1)
RUN_ID = "dfl_lomo_20261111_01"
OUT = ROOT / "model/영상추출/추출csv/dead_ball" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
NOTEBOOK = ROOT / "model/영상추출/모델/dead_ball/dfl_model_comparison.ipynb"
TRAINING = {match: (index + SEED) for index, match in enumerate(MATCHES)}
TREE_NAMES = ("LightGBM", "XGBoost", "CatBoost")
TEMPORAL_NAMES = ("TCN 2s", "TCN 4s")
EPOCHS = 6
CHUNK = 256
WIDTH = 24
DELTA_GATE = 0.01
WIN_GATE = 5
VITERBI = CONFIG["smooth_choice"]
assert len(MATCHES) == 7 and len(set(MATCHES)) == 7 and FPS == 25 and STRIDE == 5

## 데이터 준비

트리 입력은 HSJ v2의 264개 열. 시계열 입력은 v2의 현재 시점 24개 특징과 결측 표시. 학습은 5프레임 간격으로 샘플링한다. 검증 점수는 25fps에서 계산한다. 공 좌표는 라인아웃 후처리에만 쓴다.

In [ ]:
def digest(path):
    checksum = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            checksum.update(block)
    return checksum.hexdigest()


def context_columns(match):
    import pyarrow.parquet as parquet

    path = Path(train_dfl.feature_dir()) / f"{match}_rand_{TRAINING[match]}.parquet"
    return train_dfl.model_columns(parquet.ParquetFile(path).schema_arrow.names)


COLUMNS = context_columns(MATCHES[0])
BASE = [column for column in COLUMNS if "@" not in column]
BASE_INDEX = np.array([COLUMNS.index(column) for column in BASE])
assert len(COLUMNS) == 264 and len(BASE) == 24


def prepare_inputs():
    started = perf_counter()
    train = {}
    for match, seed in TRAINING.items():
        sampled = train_dfl.match_features(match, "rand", seed).iloc[::STRIDE]
        train[match] = (
            sampled[COLUMNS].to_numpy(dtype=np.float32, copy=True),
            sampled["dead"].to_numpy(dtype=np.int8, copy=True),
            sampled["period"].to_numpy(dtype=np.int8, copy=True),
        )
        del sampled
        checked = train_dfl.match_features(match, "sim", SEED)
        assert set(checked["dead"].unique()) == {0, 1}
        assert set(checked["period"].unique()) == {1, 2}
        print(f"준비 {match}: 검증 {len(checked):,}프레임", flush=True)
        del checked
    return train, perf_counter() - started


def validation_input(match, columns):
    cache = Path(train_dfl.feature_dir()) / f"{match}_sim_{SEED}.parquet"
    frame = pd.read_parquet(cache, columns=[*columns, "dead", "period"])
    tracking, events, meta = dfl_load.load_match(match)
    label = dfl_labels.dead_label(tracking)
    period = tracking["period"].to_numpy()
    np.testing.assert_array_equal(frame["dead"], label)
    np.testing.assert_array_equal(frame["period"], period)
    geometry = final_dead_balls.ball_signals(features.match_from_dfl(tracking, meta))
    crossings = final_dead_balls.line_crossings(geometry)
    spells = dfl_labels.dead_spells(tracking, events)
    return frame[columns].to_numpy(dtype=np.float32, copy=True), frame[["dead", "period"]], geometry, crossings, spells

## 모델과 채점

트리 세 모델은 동일 v2 특징·학습 프레임, 최대 400회 부스팅. TCN은 2초와 4초 고정 사양을 각각 평가한다. 창과 설정은 검증 경기의 정답으로 조정하지 않는다. 시계열 모델은 학습 경기 통계로만 정규화한다.

In [ ]:
def fit_tree(name, train_data):
    if name == "LightGBM":
        original_stride = CONFIG["train_stride"]
        original_params = CONFIG["lgbm_params"]
        try:
            CONFIG["train_stride"] = 1  # 입력에서 기존 stride를 이미 적용
            CONFIG["lgbm_params"] = dict(original_params, num_threads=THREADS)
            frames = [pd.DataFrame(x, columns=COLUMNS).assign(dead=y) for x, y, _ in train_data]
            model, _ = train_dfl.train(frames)
        finally:
            CONFIG["train_stride"] = original_stride
            CONFIG["lgbm_params"] = original_params
        return lambda values: model.predict(values, num_threads=THREADS)

    matrix = np.concatenate([x for x, _, _ in train_data])
    target = np.concatenate([y for _, y, _ in train_data])
    if name == "XGBoost":
        import xgboost as xgb

        data = xgb.QuantileDMatrix(matrix, label=target, max_bin=127, nthread=THREADS)
        model = xgb.train(
            dict(objective="binary:logistic", eta=0.05, tree_method="hist",
                 grow_policy="lossguide", max_depth=0, max_leaves=63, max_bin=127,
                 min_child_weight=25, subsample=0.8, colsample_bytree=0.7,
                 reg_lambda=1.0, seed=0, nthread=THREADS, verbosity=0),
            data, num_boost_round=400,
        )
        return model.inplace_predict
    elif name == "CatBoost":
        from catboost import CatBoostClassifier

        model = CatBoostClassifier(
            iterations=400, learning_rate=0.05, depth=6, border_count=64,
            l2_leaf_reg=3.0, random_seed=0, thread_count=THREADS,
            allow_writing_files=False, verbose=False,
        )
    else:
        raise ValueError(name)
    model.fit(matrix, target)
    return lambda values: model.predict_proba(values)[:, 1]


def period_slices(period):
    ends = np.r_[0, np.flatnonzero(period[1:] != period[:-1]) + 1, len(period)]
    return [slice(a, b) for a, b in zip(ends[:-1], ends[1:])]


def normalized(values, mean, scale):
    missing = ~np.isfinite(values)
    clean = np.clip((values - mean) / scale, -12, 12)
    clean[missing] = 0
    return np.concatenate((clean, missing.astype(np.float32)), axis=1)


def tcn_chunks(values, radius):
    tail = (-len(values)) % CHUNK
    padded = np.pad(values, ((radius, radius + tail), (0, 0)), mode="edge")
    index = np.arange(0, len(values), CHUNK)[:, None] + np.arange(CHUNK + 2 * radius)
    return np.ascontiguousarray(padded[index].transpose(0, 2, 1))


def fit_temporal(name, train_data):
    import torch
    from torch import nn

    torch.manual_seed(17)
    torch.set_num_threads(min(THREADS, 4))
    rng = np.random.default_rng(17)
    horizon = int(name.split()[1].removesuffix("s"))
    radius = horizon * FPS // (2 * STRIDE)
    dilations = (1, 2, 2) if horizon == 2 else (1, 2, 3, 4)
    assert radius == sum(dilations)
    base = np.concatenate([x[:, BASE_INDEX] for x, _, _ in train_data])
    mean = np.nanmean(base, axis=0)
    scale = np.maximum(np.nanstd(base, axis=0), 1e-3)
    assert np.isfinite(mean).all() and np.isfinite(scale).all()
    del base
    layers = [nn.Conv1d(len(BASE) * 2, WIDTH, 1), nn.GELU()]
    for dilation in dilations:
        layers.extend([nn.Conv1d(WIDTH, WIDTH, 3, dilation=dilation), nn.GELU()])
    layers.append(nn.Conv1d(WIDTH, 1, 1))
    model = nn.Sequential(*layers)

    examples, labels = [], []
    for x, y, period in train_data:
        values = normalized(x[:, BASE_INDEX], mean, scale)
        for half in period_slices(period):
            examples.append(tcn_chunks(values[half], radius))
            padded = np.pad(y[half].astype(np.float32), (0, (-len(y[half])) % CHUNK), constant_values=-1)
            labels.append(padded.reshape(-1, CHUNK))
    train_x = torch.from_numpy(np.concatenate(examples))
    train_y = torch.from_numpy(np.concatenate(labels))
    del examples, labels

    optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.001)
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    model.train()
    for _ in range(EPOCHS):
        for indices in np.array_split(rng.permutation(len(train_x)), max(1, (len(train_x) + 15) // 16)):
            optimizer.zero_grad(set_to_none=True)
            logits = model(train_x[indices]).squeeze(1)
            target = train_y[indices]
            mask = target >= 0
            loss = (criterion(logits, target.clamp_min(0)) * mask).sum() / mask.sum()
            loss.backward()
            optimizer.step()
    model.eval()

    def predict(values, period):
        score = np.empty(len(values), dtype=np.float32)
        values = normalized(values, mean, scale)
        with torch.inference_mode():
            for half in period_slices(period):
                for offset in range(STRIDE):
                    rows = slice(half.start + offset, half.stop, STRIDE)
                    if not len(values[rows]):
                        continue
                    batches = torch.from_numpy(tcn_chunks(values[rows], radius))
                    logits = torch.cat([model(batch).reshape(-1) for batch in batches.split(32)])
                    score[rows] = expit(logits.numpy()[:len(values[rows])])
        return score

    return predict


def count_false_dead(state, label, period):
    false_dead = 0
    for half in period_slices(period):
        for start, stop in features.runs_of(state[half] == 1):
            false_dead += int(label[half][start:stop].mean() < 0.5)
    return false_dead


def score_states(score, labels, geometry, crossings, spells):
    period = labels["period"].to_numpy()
    label = labels["dead"].to_numpy()
    free_entry = np.zeros(len(label), dtype=bool)
    free_entry[crossings] = True
    base = smoothing.viterbi(score, period, VITERBI["penalty"], VITERBI["bias"])
    free = smoothing.viterbi(score, period, VITERBI["penalty"], VITERBI["bias"], free_entry=free_entry)
    forced, _ = final_dead_balls.force_line_outs(free, crossings, geometry, score, period)
    result = []
    for stage, state in (("viterbi", base), ("line_out", forced)):
        row = scoring.score_one(state, labels)
        segments = metrics.segment_metrics(spells, state, period)
        found = segments["found"]
        row.update(
            stage=stage,
            false_dead=count_false_dead(state, label, period),
            missed_dead=int((~found).sum()),
            missed_free_kick=int(((spells["cause"].to_numpy() == "free_kick") & ~found).sum()),
            start_abs_med_s=float(segments["start_abs_med"]),
            end_abs_med_s=float(segments["end_abs_med"]),
            true_dead=int(len(spells)),
        )
        result.append(row)
    return result


def run_models(names, train, rows):
    for name in names:
        temporal = name in TEMPORAL_NAMES
        for held in MATCHES:
            already = [row for row in rows if row["model"] == name and row["match"] == held]
            if len(already) == 2:
                continue
            assert not already, (name, held)
            started = perf_counter()
            train_data = [train[match] for match in MATCHES if match != held]
            evaluation = validation_input(held, BASE if temporal else COLUMNS)
            values, labels, geometry, crossings, spells = evaluation
            fit_start = perf_counter()
            if temporal:
                predict = fit_temporal(name, train_data)
            else:
                predict = fit_tree(name, train_data)
            fit_s = perf_counter() - fit_start
            inference_start = perf_counter()
            score = predict(values, labels["period"].to_numpy()) if temporal else predict(values)
            infer_s = perf_counter() - inference_start
            assert score.shape == (len(labels),) and np.isfinite(score).all()
            assert np.all((0 <= score) & (score <= 1))
            outcomes = score_states(score, labels, geometry, crossings, spells)
            fold_s = perf_counter() - started
            rows.extend(dict(model=name, match=held, fit_s=fit_s, predict_s=infer_s, fold_s=fold_s, **row)
                        for row in outcomes)
            pending = OUT / "per_match.tmp"
            pd.DataFrame(rows).to_csv(pending, index=False)
            pending.replace(OUT / "per_match.csv")
            print(f"{name} | {held} | 학습 {fit_s:.1f}s | 전체 {fold_s:.1f}s | "
                  f"F1 {outcomes[0]['alive_spell_f1']:.3f} / {outcomes[1]['alive_spell_f1']:.3f}", flush=True)
            del predict, values, evaluation, labels, geometry, crossings, spells, score, outcomes

## 실행 및 결과

총 소요 시간은 각 검증 경기의 데이터 읽기·학습·예측·채점을 합산한다. 공통 준비 시간은 별도 기록한다. 가짜 데드볼은 예측 데드 구간의 절반 미만이 정답 D에 겹칠 때, 누락은 정답 데드 구간의 절반 미만을 맞혔을 때 센다. 경계 오차는 HSJ 정의를 사용한다.

**추가 실험 기준** · TCN이 라인아웃 적용 전 인플레이 구간 F1을 0.010 이상 높이고 7경기 중 5경기 이상 개선하면 Transformer 비교를 검토한다. GRU는 TCN 결과에서 반복적으로 개선할 오류가 확인될 때 후속으로 검토한다.

In [ ]:
def write_reports(rows, preparation_s):
    table = pd.DataFrame(rows)
    expected = {(model, match, stage) for model in (*TREE_NAMES, *TEMPORAL_NAMES)
                for match in MATCHES for stage in ("viterbi", "line_out")}
    observed = set(table[["model", "match", "stage"]].itertuples(index=False, name=None))
    assert observed == expected and len(table) == len(expected)
    assert table["alive_spell_f1"].between(0, 1).all()
    assert table["dead_f1"].between(0, 1).all()
    assert (table["missed_dead"] <= table["true_dead"]).all()
    by_fold = table.pivot(index=["match", "stage"], columns="model", values="alive_spell_f1")
    baseline = by_fold["LightGBM"]
    summaries = []
    for (model, stage), group in table.groupby(["model", "stage"], sort=False):
        delta = (by_fold[model] - baseline).xs(stage, level="stage")
        summaries.append(dict(
            model=model, stage=stage, matches=len(group), alive_spell_f1=group.alive_spell_f1.mean(),
            delta_vs_lightgbm=delta.mean(), wins_vs_lightgbm=int((delta > 1e-9).sum()),
            dead_frame_f1=group.dead_f1.mean(), false_dead=int(group.false_dead.sum()),
            missed_dead=int(group.missed_dead.sum()), missed_free_kick=int(group.missed_free_kick.sum()),
            start_abs_med_s=group.start_abs_med_s.mean(), end_abs_med_s=group.end_abs_med_s.mean(),
            fit_s=group.fit_s.sum(), predict_s=group.predict_s.sum(), fold_s=group.fold_s.sum(),
        ))
    summary = pd.DataFrame(summaries)
    summary.to_csv(OUT / "model_summary.csv", index=False)
    temporal = summary[(summary.model.isin(TEMPORAL_NAMES)) & (summary.stage == "viterbi")]
    transformer_gate = bool(((temporal.delta_vs_lightgbm >= DELTA_GATE) & (temporal.wins_vs_lightgbm >= WIN_GATE)).any())
    cache = Path(train_dfl.feature_dir())
    assets = [cache / f"{m}_rand_{TRAINING[m]}.parquet" for m in MATCHES]
    assets += [cache / f"{m}_sim_{SEED}.parquet" for m in MATCHES]
    manifest = dict(
        run_id=RUN_ID, code_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        notebook_sha256=digest(NOTEBOOK), python=platform.python_version(), platform=platform.platform(),
        packages={name: version(name) for name in ("numpy", "pandas", "scipy", "pyarrow", "lightgbm", "xgboost", "catboost", "torch")},
        input_feature_sha256={str(path.relative_to(ROOT)): digest(path) for path in assets},
        matches=MATCHES, training_seed_by_match=TRAINING, evaluation_simulation_seed=SEED, train_stride=STRIDE,
        fps=FPS, threads=THREADS, temporal_epochs=EPOCHS, temporal_chunk=CHUNK, temporal_width=WIDTH,
        viterbi=VITERBI, transformer_gate=transformer_gate, common_preparation_s=preparation_s,
        output_sha256={name: digest(OUT / name) for name in ("per_match.csv", "model_summary.csv")},
    )
    (OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    (OUT / "validation.json").write_text(json.dumps(dict(passed=True, expected_rows=len(expected),
        actual_rows=len(table), matches=len(MATCHES), stages=["viterbi", "line_out"]), indent=2) + "\n", encoding="utf-8")
    print(summary.round(3).to_string(index=False), flush=True)
    print(f"Transformer gate: {transformer_gate}", flush=True)
    return summary


train, preparation_s = prepare_inputs()
checkpoint = OUT / "per_match.csv"
rows = pd.read_csv(checkpoint).to_dict("records") if checkpoint.is_file() else []
stage = os.environ.get("DFL13_STAGE", "all")
assert stage in ("prepare", "trees", "temporal", "all")
if stage in ("trees", "all"):
    run_models(TREE_NAMES, train, rows)
if stage in ("temporal", "all"):
    run_models(TEMPORAL_NAMES, train, rows)
if len(rows) == 2 * len(MATCHES) * (len(TREE_NAMES) + len(TEMPORAL_NAMES)):
    summary = write_reports(rows, preparation_s)